# 02 Collector health check (project 1.3)

The notebook version of `check.sh`, with tables and charts. **Read-only**: it never changes `data_raw*/` and never calls the LTA API, so it is safe to run while the collectors are running.

| Collector | Service (Linux) | Output folder |
|---|---|---|
| Whole network + incidents, VMS, travel times, road works (`LTA_ACCOUNT_KEY`) | `lta-collector` | `data_raw/` |
| Expressway pages every 2 min (`LTA_ACCOUNT_KEY2`) | `lta-collector-expressway` | `data_raw_expressway/` |

Run top to bottom with the kernel **Python (SLDS 1.3)**, from the project folder. Change `HOURS` in the first code cell to look further back. On Windows the service section is skipped; use `01_collector_monitor.ipynb` there.

What healthy looks like, per hour: `data_raw` about 12 speed-band fetches; `data_raw_expressway` about 30; no `probe_failed` or `fetch_failed`; heartbeats under 2 minutes; expressway coverage close to 100% for every update older than about 20 minutes (LTA rolls each update out over up to ~15 minutes).

AI disclosure: drafted with Claude (Anthropic), 16 Oct 2026; log in `AI_USE.md`.

## 1. Settings

In [ ]:
import json, platform, shutil, subprocess
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

HOURS = 6                         # how far back to look
SETTLE_MIN = 20                   # LTA rolls an update out over up to ~15 min; newer updates are left out of coverage
PROJ = Path.cwd()
SGT = timezone(timedelta(hours=8))
COLLECTORS = {"data_raw": "lta-collector", "data_raw_expressway": "lta-collector-expressway"}

now = datetime.now(SGT)
since = now - timedelta(hours=HOURS)
print(f"Now {now:%Y-%m-%d %H:%M} SGT; looking back {HOURS} h, to {since:%Y-%m-%d %H:%M}")
missing = [d for d in COLLECTORS if not (PROJ / d).is_dir()]
if missing:
    print("Folders not found (wrong working directory, or that collector never ran):", missing)

## 2. Services (Linux only)

In [ ]:
def systemctl_show(unit: str) -> dict:
    out = subprocess.run(["systemctl", "show", unit, "-p", "ActiveState", "-p", "NRestarts",
                          "-p", "ActiveEnterTimestamp", "-p", "UnitFileState"],
                         capture_output=True, text=True).stdout
    return dict(line.split("=", 1) for line in out.splitlines() if "=" in line)

if platform.system() == "Linux" and shutil.which("systemctl"):
    rows = []
    for folder, unit in COLLECTORS.items():
        s = systemctl_show(unit)
        rows.append({"service": unit, "active": s.get("ActiveState"), "at boot": s.get("UnitFileState"),
                     "restarts": s.get("NRestarts"), "running since": s.get("ActiveEnterTimestamp")})
    print(pl.DataFrame(rows))
else:
    print("Not Linux: skipped. On the Windows desktop use 01_collector_monitor.ipynb, cell 5.")

## 3. Status of each collector

From `collector_status.json`, which each collector rewrites every 30 seconds.

In [ ]:
for folder in COLLECTORS:
    path = PROJ / folder / "collector_status.json"
    print(f"== {folder} ==")
    if not path.exists():
        print("  no status file yet\n")
        continue
    s = json.loads(path.read_text(encoding="utf-8"))
    beat = datetime.strptime(s["last_heartbeat"], "%Y-%m-%dT%H:%M:%S%z")
    age = (now - beat).total_seconds() / 60
    print(f"  heartbeat   {age:.1f} min ago  ({'OK' if age < 2 else 'PROBLEM: collector stopped or frozen'})")
    print(f"  started     {s.get('started')}")
    print(f"  today       {s.get('today')}")
    print(f"  last error  {s.get('last_error')}\n")

## 4. Speed-band fetches per hour

One row per fetch in each `log_speedbands.csv`. `ok_flagged` is normal (LTA's pages carry mixed update times). `probe_failed` or `fetch_failed` with "Rate limit" means a key hit LTA's hourly limit (about 4,000 calls).

In [ ]:
def read_log(folder: str) -> pl.DataFrame:
    path = PROJ / folder / "log_speedbands.csv"
    if not path.exists():
        return pl.DataFrame()
    df = pl.read_csv(path, infer_schema_length=0, null_values="NA")
    return (df.with_columns(pl.col("retrieved_at").str.to_datetime("%Y-%m-%dT%H:%M:%S%z").dt.convert_time_zone("Asia/Singapore").dt.replace_time_zone(None).alias("t"),
                            pl.col("pages").cast(pl.Int64, strict=False))
              .filter(pl.col("t") >= since.replace(tzinfo=None))   # naive Singapore time on both sides
              .with_columns(pl.col("t").dt.truncate("1h").dt.strftime("%m-%d %H:00").alias("hour"),
                            pl.col("message").fill_null("").str.contains("Rate limit").alias("rate_limited")))

logs = {folder: read_log(folder) for folder in COLLECTORS}
for folder, df in logs.items():
    print(f"== {folder}: fetches per hour by status ==")
    if df.is_empty():
        print("  none in the window\n")
        continue
    print(df.pivot(on="status", index="hour", values="t", aggregate_function="len").sort("hour").fill_null(0))
    extra = " (plus about 60 one-page probes an hour)" if folder == "data_raw" else ""
    print(f"  pages requested in the window: {df['pages'].sum():,}{extra}")
    rl = df.filter(pl.col("rate_limited"))
    print(f"  rate-limit errors: {rl.height}" + (f", first {rl['t'].min():%H:%M}, last {rl['t'].max():%H:%M}" if rl.height else ""))
    print()

## 5. Expressway coverage per LTA update

For each 5-minute LTA update: how many of the expressway links were captured, by `data_raw` alone and by both collectors together. The list of expressway links comes from the expressway collector's page map (`data_raw_expressway/collector_state.json`). This is what the project needs: the "unusual" label uses consecutive updates per link, so gaps here become missing rows.

In [ ]:
state_path = PROJ / "data_raw_expressway" / "collector_state.json"
xw = set(json.loads(state_path.read_text(encoding="utf-8")).get("xw_links") or []) if state_path.exists() else set()
print(f"Expressway links in the page map: {len(xw):,}")

def files_in_window(folder: str) -> list[Path]:
    days = {(since + timedelta(days=i)).strftime("%Y-%m-%d") for i in range((now - since).days + 2)}
    out = []
    for day in sorted(days):
        for f in sorted((PROJ / folder / "speedbands" / day).glob("sb_*.csv.gz")):
            t = datetime.strptime(f.name[3:18], "%Y%m%d_%H%M%S").replace(tzinfo=SGT)
            if t >= since - timedelta(minutes=15):   # a fetch can carry updates up to ~15 min old
                out.append(f)
    return out

def links_per_update(files: list[Path]) -> pl.DataFrame:
    if not files or not xw:
        return pl.DataFrame(schema={"last_updated": pl.Utf8, "links": pl.UInt32})
    frames = [pl.read_csv(f, columns=["last_updated", "LinkID"], schema_overrides={"LinkID": pl.Utf8, "last_updated": pl.Utf8})
                .filter(pl.col("LinkID").is_in(list(xw))) for f in files]
    return pl.concat(frames).unique().group_by("last_updated").agg(pl.len().alias("links"))

f1, f2 = files_in_window("data_raw"), files_in_window("data_raw_expressway")
lo = since.strftime("%Y-%m-%d %H:%M:%S")
hi = (now - timedelta(minutes=SETTLE_MIN)).strftime("%Y-%m-%d %H:%M:%S")
cov = (links_per_update(f1 + f2).rename({"links": "both"})
       .join(links_per_update(f1).rename({"links": "data_raw_only"}), on="last_updated", how="left")
       .filter(pl.col("last_updated").is_between(pl.lit(lo), pl.lit(hi)))
       .with_columns(pl.col("data_raw_only").fill_null(0))
       .with_columns((pl.col("data_raw_only") / len(xw) * 100).round(1).alias("pct_data_raw"),
                     (pl.col("both") / len(xw) * 100).round(1).alias("pct_both"))
       .sort("last_updated"))
if cov.is_empty():
    print("No complete updates in the window yet.")
else:
    expected = int((now - timedelta(minutes=SETTLE_MIN) - since).total_seconds() // 300)
    print(f"Updates seen: {cov.height} of about {expected} expected in the window")
    print(f"Mean coverage: data_raw alone {cov['pct_data_raw'].mean():.1f}%, both collectors {cov['pct_both'].mean():.1f}%")
    print(f"Updates under 95% with both collectors: {(cov['pct_both'] < 95).sum()}")
    with pl.Config(tbl_rows=12):
        print(cov.tail(12))

In [ ]:
if not cov.is_empty():
    t = cov["last_updated"].str.to_datetime("%Y-%m-%d %H:%M:%S")
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(t, cov["pct_both"], marker="o", ms=3, label="both collectors")
    ax.plot(t, cov["pct_data_raw"], marker="x", ms=3, alpha=0.7, label="data_raw alone")
    ax.axhline(95, color="grey", lw=0.8, ls="--")
    ax.set_ylim(0, 102)
    ax.set_ylabel("% of expressway links")
    ax.set_title(f"Expressway links captured per LTA update, last {HOURS} h")
    ax.legend(loc="lower left")
    fig.autofmt_xdate()
    plt.show()

## 6. Disk

In [ ]:
def folder_size_gb(p: Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9 if p.exists() else 0.0

usage = shutil.disk_usage(PROJ)
print(f"Free: {usage.free / 1e9:,.0f} GB of {usage.total / 1e9:,.0f} GB")
for folder in COLLECTORS:
    print(f"{folder}: {folder_size_gb(PROJ / folder):.2f} GB")